Importamos el dataset desde kaggle (doc de kaggle)


In [12]:
import kagglehub

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType,
    LongType,
    DoubleType,
    DateType
)

from pyspark.sql.window import Window

import re

from pyspark.sql import functions as F
from pyspark.sql.types import StringType, DecimalType

In [6]:

dataset_path = kagglehub.dataset_download(
    "algorismus/adventure-works-in-excel-tables"
)


100%|██████████| 554k/554k [00:00<00:00, 95.1MB/s]

Extracting files...
/root/.cache/kagglehub/datasets/algorismus/adventure-works-in-excel-tables/versions/6


In [9]:
print(dataset_path)

/root/.cache/kagglehub/datasets/algorismus/adventure-works-in-excel-tables/versions/6


In [8]:
spark = (
    SparkSession.builder
    .appName("AdventureWorksAnalysis")
    .getOrCreate()
)

In [37]:
sales_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .option("sep", "\t")
    .csv(f"{dataset_path}/Sales.csv")
)

region_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .option("sep", "\t")
    .csv(f"{dataset_path}/Region.csv")
)

product_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .option("sep", "\t")
    .csv(f"{dataset_path}/Product.csv")
)

salesperson_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .option("sep", "\t")
    .csv(f"{dataset_path}/Salesperson.csv")
)

reseller_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .option("sep", "\t")
    .csv(f"{dataset_path}/Reseller.csv")
)

salesperson_region_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .option("sep", "\t")
    .csv(f"{dataset_path}/SalespersonRegion.csv")
)

targets_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .option("sep", "\t")
    .csv(f"{dataset_path}/Targets.csv")
)

In [18]:
print(sales_df)

DataFrame[salesordernumber: string, orderdate: string, productkey: int, resellerkey: int, employeekey: int, salesterritorykey: int, quantity: int, unit_price: string, sales: string, cost: string]


Funciones para limpieza de datos


In [35]:

def normalize_column_names(df):
    for column in df.columns:

        new_name = re.sub(
            r"([a-z0-9])([A-Z])",
            r"\1_\2",
            column
        )

        new_name = re.sub(
            r"[^a-zA-Z0-9]+",
            "_",
            new_name
        )

        new_name = new_name.strip("_").lower()

        df = df.withColumnRenamed(
            column,
            new_name
        )

    return df

In [14]:
def clean_strings(df):
    for field in df.schema.fields:
        if isinstance(field.dataType, StringType):
            df = df.withColumn(
                field.name,
                F.when(
                    F.trim(F.col(field.name)) == "",
                    None
                ).otherwise(
                    F.trim(F.col(field.name))
                )
            )

    return df

In [15]:
def clean_money(column_name):
    return (
        F.regexp_replace(
            F.col(column_name),
            r"[$,]",
            ""
        )
        .cast(DecimalType(18, 2))
    )

Normalizamos y limpiamos los dataframes

In [38]:
sales_df = normalize_column_names(sales_df)
sales_df = clean_strings(sales_df)

sales_df = (
    sales_df
    .dropDuplicates()

    .withColumn(
        "order_date",
        F.regexp_replace(
            F.col("order_date"),
            r"^[A-Za-z]+,\s*",
            ""
        )
    )

    .withColumn(
        "order_date",
        F.to_date(
            F.col("order_date"),
            "MMMM d, yyyy"
        )
    )

    .withColumn(
        "unit_price",
        clean_money("unit_price")
    )

    .withColumn(
        "sales",
        clean_money("sales")
    )

    .withColumn(
        "cost",
        clean_money("cost")
    )

    .withColumn(
        "quantity",
        F.col("quantity").cast("int")
    )

    .withColumn(
        "product_key",
        F.col("product_key").cast("int")
    )

    .withColumn(
        "reseller_key",
        F.col("reseller_key").cast("int")
    )

    .withColumn(
        "employee_key",
        F.col("employee_key").cast("int")
    )

    .withColumn(
        "sales_territory_key",
        F.col("sales_territory_key").cast("int")
    )
)

In [40]:
product_df = normalize_column_names(product_df)
product_df = clean_strings(product_df)

product_df = (
    product_df
    .withColumn(
        "product_key",
        F.col("product_key").cast("int")
    )
    .withColumn(
        "standard_cost",
        clean_money("standard_cost")
    )
    .dropDuplicates(["product_key"])
)

In [41]:
salesperson_df = normalize_column_names(salesperson_df)
salesperson_df = clean_strings(salesperson_df)

salesperson_df = (
    salesperson_df
    .withColumn(
        "employee_key",
        F.col("employee_key").cast("int")
    )
    .withColumn(
        "employee_id",
        F.col("employee_id").cast("long")
    )
    .dropDuplicates(["employee_key"])
)

In [42]:
region_df = normalize_column_names(region_df)
region_df = clean_strings(region_df)

region_df = (
    region_df
    .withColumn(
        "sales_territory_key",
        F.col("sales_territory_key").cast("int")
    )
    .dropDuplicates(["sales_territory_key"])
)

In [43]:
salesperson_region_df = normalize_column_names(
    salesperson_region_df
)

salesperson_region_df = (
    salesperson_region_df
    .withColumn(
        "employee_key",
        F.col("employee_key").cast("int")
    )
    .withColumn(
        "sales_territory_key",
        F.col("sales_territory_key").cast("int")
    )
    .dropDuplicates([
        "employee_key",
        "sales_territory_key"
    ])
)

In [44]:
reseller_df = normalize_column_names(reseller_df)
reseller_df = clean_strings(reseller_df)

reseller_df = (
    reseller_df
    .withColumn(
        "reseller_key",
        F.col("reseller_key").cast("int")
    )
    .dropDuplicates(["reseller_key"])
)

In [45]:
targets_df = normalize_column_names(targets_df)
targets_df = clean_strings(targets_df)

targets_df = (
    targets_df
    .withColumn(
        "employee_id",
        F.col("employee_id").cast("long")
    )
    .withColumn(
        "target",
        clean_money("target")
    )
    .withColumn(
        "target_month",
        F.regexp_replace(
            F.col("target_month"),
            r"^[A-Za-z]+,\s*",
            ""
        )
    )
    .withColumn(
        "target_month",
        F.to_date(
            F.col("target_month"),
            "MMMM d, yyyy"
        )
    )
    .dropDuplicates([
        "employee_id",
        "target_month"
    ])
)

Validaciones

In [32]:
def validate_df(name, df):
    total_rows = df.count()
    duplicate_rows = total_rows - df.dropDuplicates().count()

    print(f"{name}")
    print(f"Total rows: {total_rows}")
    print(f"Duplicate rows: {duplicate_rows}")

    df.select(
        [
            F.sum(
                F.when(
                    F.col(column).isNull(),
                    1
                ).otherwise(0)
            ).alias(column)
            for column in df.columns
        ]
    ).show()

In [39]:
validate_df("Sales", sales_df)


Sales
Total rows: 57851
Duplicate rows: 0
+------------------+----------+-----------+------------+------------+-------------------+--------+----------+-----+----+
|sales_order_number|order_date|product_key|reseller_key|employee_key|sales_territory_key|quantity|unit_price|sales|cost|
+------------------+----------+-----------+------------+------------+-------------------+--------+----------+-----+----+
|                 0|         0|          0|           0|           0|                  0|       0|         0|    0|   0|
+------------------+----------+-----------+------------+------------+-------------------+--------+----------+-----+----+



In [46]:
validate_df("Product", product_df)


Product
Total rows: 397
Duplicate rows: 0
+-----------+-------+-------------+-----+-----------+--------+-----------------------+-----------------+
|product_key|product|standard_cost|color|subcategory|category|background_color_format|font_color_format|
+-----------+-------+-------------+-----+-----------+--------+-----------------------+-----------------+
|          0|      0|            0|    0|          0|       0|                      0|                0|
+-----------+-------+-------------+-----+-----------+--------+-----------------------+-----------------+



In [47]:
validate_df("Salesperson", salesperson_df)


Salesperson
Total rows: 18
Duplicate rows: 0
+------------+-----------+-----------+-----+---+
|employee_key|employee_id|salesperson|title|upn|
+------------+-----------+-----------+-----+---+
|           0|          0|          0|    0|  0|
+------------+-----------+-----------+-----+---+



In [48]:
validate_df("Region", region_df)


Region
Total rows: 10
Duplicate rows: 0
+-------------------+------+-------+-----+
|sales_territory_key|region|country|group|
+-------------------+------+-------+-----+
|                  0|     0|      0|    0|
+-------------------+------+-------+-----+



In [49]:
validate_df("SalespersonRegion", salesperson_region_df)


SalespersonRegion
Total rows: 39
Duplicate rows: 0
+------------+-------------------+
|employee_key|sales_territory_key|
+------------+-------------------+
|           0|                  0|
+------------+-------------------+



In [50]:
validate_df("Reseller", reseller_df)


Reseller
Total rows: 701
Duplicate rows: 0
+------------+-------------+--------+----+--------------+--------------+
|reseller_key|business_type|reseller|city|state_province|country_region|
+------------+-------------+--------+----+--------------+--------------+
|           0|            0|       0|   0|             0|             0|
+------------+-------------+--------+----+--------------+--------------+



In [51]:
validate_df("Targets", targets_df)

Targets
Total rows: 809
Duplicate rows: 0
+-----------+------+------------+
|employee_id|target|target_month|
+-----------+------+------------+
|          0|     0|           0|
+-----------+------+------------+



Consultas

Mostrar las ventas por región comparado anualmente

In [52]:
sales_by_region = (
    sales_df
    .join(
        region_df.select(
            "sales_territory_key",
            "region"
        ),
        on="sales_territory_key",
        how="inner"
    )
    .withColumn(
        "year",
        F.year("order_date")
    )
    .groupBy("region")
    .pivot(
        "year",
        [2017, 2018, 2019, 2020]
    )
    .agg(
        F.sum("sales")
    )
    .fillna(0)
    .withColumnRenamed("2017", "Año_2017")
    .withColumnRenamed("2018", "Año_2018")
    .withColumnRenamed("2019", "Año_2019")
    .withColumnRenamed("2020", "Año_2020")
    .orderBy("region")
)

sales_by_region.show(truncate=False)

+--------------+----------+----------+----------+----------+
|region        |Año_2017  |Año_2018  |Año_2019  |Año_2020  |
+--------------+----------+----------+----------+----------+
|Australia     |0.00      |0.00      |875621.82 |515403.03 |
|Canada        |1514285.89|4862018.61|5691150.09|1808178.37|
|Central       |951799.96 |2635728.33|3016263.54|1029595.03|
|France        |0.00      |860904.52 |2406134.06|1260801.11|
|Germany       |0.00      |0.00      |1127594.53|750148.86 |
|Northeast     |568552.28 |2453772.16|2876166.60|816863.35 |
|Northwest     |1690237.48|3502591.03|4687348.50|2124645.44|
|Southeast     |1450813.30|2838535.04|2444552.75|904705.46 |
|Southwest     |1894066.67|6330990.61|7188401.70|2587657.12|
|United Kingdom|0.00      |844245.95 |2186771.99|852025.02 |
+--------------+----------+----------+----------+----------+



Identificar los dos productos con más cantidades vendidas en cada año

In [55]:
product_sales_by_year = (
    sales_df
    .join(
        product_df.select(
            "product_key",
            "product"
        ),
        on="product_key",
        how="inner"
    )
    .withColumn(
        "year",
        F.year("order_date")
    )
    .groupBy(
        "year",
        "product_key",
        "product"
    )
    .agg(
        F.sum("quantity").alias("units_sold")
    )
)



In [56]:
product_window = (
    Window
    .partitionBy("year")
    .orderBy(
        F.desc("units_sold"),
        F.asc("product_key")
    )
)

In [57]:
top_2_products_by_year = (
    product_sales_by_year
    .withColumn(
        "ranking",
        F.row_number().over(product_window)
    )
    .filter(
        F.col("ranking") <= 2
    )
    .select(
        F.col("product").alias("Nombre_producto"),
        F.col("year").alias("año"),
        F.col("units_sold").alias("unidades_vendidas")
    )
    .orderBy(
        "año",
        F.desc("unidades_vendidas")
    )
)

top_2_products_by_year.show(truncate=False)

+-------------------------------+----+-----------------+
|Nombre_producto                |año |unidades_vendidas|
+-------------------------------+----+-----------------+
|Mountain Bike Socks, M         |2017|563              |
|AWC Logo Cap                   |2017|520              |
|Full-Finger Gloves, L          |2018|1942             |
|Long-Sleeve Logo Jersey, L     |2018|1432             |
|Classic Vest, S                |2019|2347             |
|Short-Sleeve Classic Jersey, XL|2019|2052             |
|Classic Vest, S                |2020|1349             |
|Short-Sleeve Classic Jersey, XL|2020|1107             |
+-------------------------------+----+-----------------+



Identificar al empleado con mayor cantidad de ventas en cada mes del año 2020

In [58]:
employee_sales_2020 = (
    sales_df
    .filter(
        F.year("order_date") == 2020
    )
    .join(
        salesperson_df.select(
            "employee_key",
            "employee_id",
            "salesperson"
        ),
        on="employee_key",
        how="inner"
    )
    .withColumn(
        "month",
        F.month("order_date")
    )
    .groupBy(
        "month",
        "employee_id",
        "salesperson"
    )
    .agg(
        F.sum("sales").alias("total_sales")
    )
)

In [59]:
employee_window = (
    Window
    .partitionBy("month")
    .orderBy(
        F.desc("total_sales"),
        F.asc("employee_id")
    )
)

In [60]:
top_employee_by_month = (
    employee_sales_2020
    .withColumn(
        "ranking",
        F.row_number().over(employee_window)
    )
    .filter(
        F.col("ranking") == 1
    )
    .select(
        F.col("employee_id").alias("Id_empleado"),
        F.col("salesperson").alias("Nombre_empleado"),
        F.col("month").alias("mes"),
        F.col("total_sales").alias("total_ventas")
    )
    .orderBy("mes")
)

top_employee_by_month.show(truncate=False)

+-----------+------------------------+---+------------+
|Id_empleado|Nombre_empleado         |mes|total_ventas|
+-----------+------------------------+---+------------+
|668991357  |Jae Pak                 |1  |219460.40   |
|191644724  |Linda Mitchell          |2  |499364.86   |
|668991357  |Jae Pak                 |3  |353918.00   |
|668991357  |Jae Pak                 |4  |370602.30   |
|134219713  |Ranjit Varkey Chudukatil|5  |608336.73   |
+-----------+------------------------+---+------------+

